# OpenAI Community Forum: engagement analytics

Reads the ClickHouse tables loaded by `etl.py` and builds four visuals:

1. **Bar chart** - top topics by views
2. **Scatter** - posts vs. topic age (velocity)
3. **Heatmap** - topic creation by weekday x hour (UTC)
4. **Bonus** - categories by posts and views

Connection settings come from the same environment variables as the ETL
(`CLICKHOUSE_HOST`, `CLICKHOUSE_PORT`, `CLICKHOUSE_USER`, `CLICKHOUSE_PASSWORD`, `CLICKHOUSE_DATABASE`).

In [ ]:
import os
import textwrap

import clickhouse_connect
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

DB = os.getenv("CLICKHOUSE_DATABASE", "discourse")
client = clickhouse_connect.get_client(
    host=os.getenv("CLICKHOUSE_HOST", "localhost"),
    port=int(os.getenv("CLICKHOUSE_PORT", "8123")),
    username=os.getenv("CLICKHOUSE_USER", "default"),
    password=os.getenv("CLICKHOUSE_PASSWORD", ""),
)


def q(sql: str) -> pd.DataFrame:
    """Run a query and return a DataFrame."""
    return client.query_df(sql)


def short(title: str, width: int = 55) -> str:
    return textwrap.shorten(title, width=width, placeholder="...")


# Sanity check: how much data do we have?
q(f"""
SELECT count() AS topics, min(created_at) AS oldest_topic, max(snapshot_ts) AS last_snapshot
FROM {DB}.v_dim_topics
""")

## 1. Top topics by views

In [ ]:
top = q(f"""
SELECT topic_id, title, views, posts_count, like_count
FROM {DB}.v_dim_topics
ORDER BY views DESC, topic_id
LIMIT 20
""")

top["label"] = top["title"].map(short)
top = top.sort_values("views")  # largest bar at the top of a horizontal chart

fig, ax = plt.subplots(figsize=(11, 8))
bars = ax.barh(top["label"], top["views"], color="#3b6fb6")
ax.bar_label(bars, labels=[f"{v:,}" for v in top["views"]], padding=3, fontsize=8)
ax.set_title("Top 20 topics by views")
ax.set_xlabel("Views")
ax.set_ylabel("")
ax.margins(x=0.12)
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

## 2. Velocity: posts vs. topic age

Each dot is a topic. The x-axis is topic age and the y-axis is its post count (both log scale);
dot size and colour encode views. Dashed lines are constant velocity (posts per day), so dots
far above a line are busier than that rate. The most active topics are labelled.

In [ ]:
vel = q(f"""
SELECT
    topic_id, title, posts_count, views,
    greatest(dateDiff('second', created_at, now64(3)) / 86400.0, 1.0 / 24) AS age_days,
    posts_count / greatest(dateDiff('second', created_at, now64(3)) / 86400.0, 1.0 / 24) AS posts_per_day
FROM {DB}.v_dim_topics
WHERE posts_count > 0
""")

fig, ax = plt.subplots(figsize=(11, 7))
sizes = 25 + 275 * vel["views"] / max(vel["views"].max(), 1)
sc = ax.scatter(vel["age_days"], vel["posts_count"], s=sizes,
                c=np.log10(vel["views"] + 1), cmap="viridis", alpha=0.7, edgecolor="none")

xs = np.logspace(np.log10(vel["age_days"].min()), np.log10(vel["age_days"].max()), 50)
for rate, style in [(1, ":"), (10, "--"), (100, "-.")]:
    ax.plot(xs, rate * xs, style, color="grey", lw=1, label=f"{rate} posts/day")

for _, r in vel.nlargest(6, "posts_per_day").iterrows():
    ax.annotate(short(r["title"], 35), (r["age_days"], r["posts_count"]),
                xytext=(6, 6), textcoords="offset points", fontsize=7)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(bottom=max(vel["posts_count"].min() * 0.8, 0.8), top=vel["posts_count"].max() * 1.6)
ax.set_xlabel("Topic age (days, log scale)")
ax.set_ylabel("Posts (log scale)")
ax.set_title("Topic velocity: posts vs. age")
ax.legend(title="Constant velocity", loc="lower right")
fig.colorbar(sc, ax=ax, label="log10(views)")
plt.tight_layout()
plt.show()

## 3. When are topics created? (weekday x hour, UTC)

In [ ]:
heat = q(f"""
SELECT toDayOfWeek(created_at) AS dow, toHour(created_at) AS hour, count() AS topics
FROM {DB}.v_dim_topics
GROUP BY dow, hour
""")

grid = (heat.pivot(index="dow", columns="hour", values="topics")
            .reindex(index=range(1, 8), columns=range(24))
            .fillna(0)
            .astype(int))
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, ax = plt.subplots(figsize=(13, 4.8))
im = ax.imshow(grid.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}" for h in range(24)])
ax.set_yticks(range(7))
ax.set_yticklabels(days)
ax.grid(False)
for (i, j), v in np.ndenumerate(grid.values):
    if v:
        ax.text(j, i, v, ha="center", va="center", fontsize=7,
                color="white" if v > grid.values.max() * 0.6 else "black")
ax.set_xlabel("Hour of day (UTC)")
ax.set_title("Topics created by weekday and hour")
fig.colorbar(im, ax=ax, label="Topics created")
plt.tight_layout()
plt.show()

## 4. Bonus: categories by posts and views

In [ ]:
cat = q(f"""
SELECT
    coalesce(c.name, concat('Category #', toString(t.category_id))) AS category,
    count() AS topics,
    sum(t.posts_count) AS total_posts,
    sum(t.views) AS total_views
FROM {DB}.v_dim_topics AS t
LEFT JOIN {DB}.v_dim_categories AS c ON t.category_id = c.category_id
GROUP BY t.category_id, c.name
ORDER BY total_views DESC
LIMIT 12
SETTINGS join_use_nulls = 1
""")

cat = cat.iloc[::-1]  # biggest at the top
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
axes[0].barh(cat["category"], cat["total_views"], color="#3b6fb6")
axes[0].set_title("Total views")
axes[1].barh(cat["category"], cat["total_posts"], color="#d9822b")
axes[1].set_title("Total posts")
for ax in axes:
    ax.grid(axis="y", visible=False)
fig.suptitle("Which categories drive engagement?")
plt.tight_layout()
plt.show()